# 07 · Tracing and eval

To compare three frameworks, every run has to be measured the same way: the same definition of a token,
an LLM call, a tool call and a second. This notebook connects LangGraph to the shared harness in
`agent_contracts.harness`, measures runs of the research workflow from notebook 05, and scores them with
the shared scoring, which is how the capstone's `run_eval.py` will work.

**You will learn**

- what the shared harness measures, and where each number comes from
- how a LangChain callback feeds model calls into it
- how to run a small, scored evaluation and write it in the shared results format

In [1]:
from agent_contracts.harness import RunMetrics, build_results, verify_model
from agent_contracts.research import tools as research
from agent_contracts.research.scoring import load_questions, score_report
from lg_agents.llm import make_llm
from lg_agents.research_flow import build_research_flow, to_report

verify_model()  # stops with the command that fixes it if Ollama or the shared model is missing
flow = build_research_flow(make_llm())  # the research workflow from notebook 05, built once and reused per question
questions = {q["id"]: q for q in load_questions()}  # the shared questions by id, each with its expected facts

## The shared definitions

| Number | Definition | Where it comes from |
|---|---|---|
| LLM calls | every model call in the run | the framework's callbacks or hooks |
| input and output tokens | summed over every model call | the counts Ollama reports with each response |
| tool calls | every attempted call, refused ones included | the shared tools record them (`research.tool_calls()`) |
| latency | wall clock for the whole question | `RunMetrics.measure()` |

Tool calls are counted inside the shared tools, so no framework can count them differently. Model calls
have to come from the framework, because only it sees them; in LangChain that is a **callback**.

In [2]:
# HarnessCallback lives in lg_agents/metrics.py. It is shown here because it is the whole bridge between
# LangChain's callbacks and the shared RunMetrics.
import inspect

import lg_agents.metrics

print(inspect.getsource(lg_agents.metrics.HarnessCallback))

class HarnessCallback(BaseCallbackHandler):
    def __init__(self, metrics: RunMetrics) -> None:
        self.metrics = metrics

    def on_llm_end(self, response, **kwargs) -> None:
        for generations in response.generations:
            for generation in generations:
                usage = getattr(getattr(generation, "message", None), "usage_metadata", None) or {}
                self.metrics.add_llm_call(usage.get("input_tokens"), usage.get("output_tokens"))



Callbacks passed in the run's `config` reach every nested model call, including the researcher agent
running inside the workflow's `research` node.

## One measured run

In [3]:
from lg_agents.metrics import HarnessCallback


def run_question(qid: str) -> dict:
    """Run one question through the workflow, measured and scored the shared way."""
    q = questions[qid]
    research.reset()  # clear the shared tools' call record, so only this question's calls are counted
    metrics = RunMetrics()
    with metrics.measure():  # times the run
        state = flow.invoke({"question_id": qid, "question": q["question"]},
                            config={"callbacks": [HarnessCallback(metrics)]})  # counts every model call
    report = to_report(state)
    scored = score_report(report, q)  # compares the report with the question's expected facts
    # One result per question, in the shared format.
    return {"id": qid, "passed": scored["passed"], "scores": scored["scores"], "problems": scored["problems"],
            "report": report.model_dump(), "tool_calls": research.tool_calls(),
            "budget_exhausted": research.budget_exhausted(), **metrics.as_dict(), "error": None}

Now run one question, `Q07_deployment`, and print what came back: whether it passed, what the run cost,
and the answer with its scores.

In [4]:
item = run_question("Q07_deployment")

print("passed:         ", item["passed"])
print("model calls:    ", item["llm_calls"])
print("tokens in / out:", item["input_tokens"], "/", item["output_tokens"])
print("time:           ", f"{item['latency_s']:.0f} s")
print("tools called:   ", ", ".join(item["tool_calls"]) or "none")
print("answer:         ", item["report"]["answer"])
print("fact recall:    ", item["scores"]["fact_recall"], "(share of the expected facts the answer contains)")
print("facts missing:  ", ", ".join(item["scores"]["facts_missing"]) or "none")
print("no stale figure:", item["scores"]["contradiction_free"])
print("citations valid:", item["scores"]["citations_valid"])
print("problems:       ", "; ".join(item["problems"]) or "none")

passed:          True
model calls:     5
tokens in / out: 2671 / 295
time:            78 s
tools called:    web_search
answer:          Veltara Aquatech has deployed 2,800 TidePure units across 14 provinces as of the end of 2025 [F04].
fact recall:     1.0 (share of the expected facts the answer contains)
facts missing:   none
no stale figure: True
citations valid: True
problems:        none


Q07 asks how many units Veltara has deployed and in how many provinces. The 2023 article still says
9 provinces; the current figure is 14. Mentioning 9 at all fails `contradiction_free`.

## A small eval

An evaluation is the same thing over a fixed set of cases, scored automatically, so you can re-run it after
every change and see whether things got better. Here are three more questions: a calculation (Q03), a
recall with a distractor article about another company (Q08), and the Series B round (Q04). Each takes a
minute or two on a 16 GB M4 Mac.

In [5]:
# Run the other three questions with the same function and keep every result in one list.
items = [item]
for qid in ("Q03_revenue_growth", "Q08_recall", "Q04_series_b"):
    items.append(run_question(qid))
    print("finished", qid)

finished Q03_revenue_growth


finished Q08_recall


finished Q04_series_b


When the runs finish, print one row per question: whether it passed, how much of the expected facts it
recalled, and what it cost.

In [6]:
# One row per question: pass or fail, recall of the expected facts, and the cost of the run.
print(f"{'question':20} {'pass':5} {'recall':6} {'calls':5} {'tokens':>7} {'secs':>5}  tools")
for it in items:
    print(f"{it['id']:20} {str(it['passed']):5} {it['scores']['fact_recall']:6} {it['llm_calls']:5} "
          f"{it['input_tokens'] + it['output_tokens']:7} {it['latency_s']:5.0f}  {', '.join(it['tool_calls'])}")

passed = sum(it["passed"] for it in items)
print(f"\n{passed} of {len(items)} questions passed")

# For any question that failed a check, show what the check found and the answer that triggered it.
for it in items:
    if it["problems"]:
        print(f"\n{it['id']} failed a check")
        print("  problems:", "; ".join(it["problems"]))
        print("  answer:  ", it["report"]["answer"])

question             pass  recall calls  tokens  secs  tools
Q07_deployment       True     1.0     5    2966    78  web_search
Q03_revenue_growth   True     1.0     6    4525   113  web_search, calculator
Q08_recall           True     1.0     6    5039   129  web_search, web_search
Q04_series_b         True     1.0     5    3139   119  web_search

4 of 4 questions passed


## The shared results format

`build_results` wraps the items in the format every framework writes, and computes the summary the same way
for all of them. The capstone's `run_eval.py` writes it to `results/research/langgraph.json`; this notebook
only prints it, so a tutorial run never overwrites real results.

In [7]:
from importlib.metadata import version

# Wrap the four results in the shared format; the framework's name and version are recorded in the file.
model_name = make_llm().model
results = build_results("research", "langgraph", version("langgraph"), model_name, items)

summary = results["summary"]  # computed the same way for every framework
print("model:        ", results["model"])
print("passed:       ", f"{summary['passed']} of {summary['total']} questions")
print("total tokens: ", summary["total_tokens"])
print("total time:   ", f"{summary['total_latency_s']:.0f} s")
print("fields saved for each question:", ", ".join(sorted(results["questions"][0])))

model:         qwen3-agents
passed:        4 of 4 questions
total tokens:  15669
total time:    439 s
fields saved for each question: budget_exhausted, error, id, input_tokens, latency_s, llm_calls, output_tokens, passed, problems, report, scores, tool_calls


## Seeing inside a run

Numbers say *how much*; traces say *what happened*. You have already used the simplest trace:
`stream_mode="updates"` (notebooks 03 and 05). LangGraph also offers `stream_mode="debug"` for every event,
and LangSmith, a hosted tracing service, which needs an account and API key and is not used in this
local-only tutorial. Whatever the tool, the traces are what you read when an eval score drops.

## How the others do this

- **CrewAI** publishes events on `crewai_event_bus`; a `BaseEventListener` subclass can collect model
  calls and token usage. Crews also report accumulated usage after a run.
- **Strands** uses hooks: `AfterModelCallEvent` and `AfterToolCallEvent` fire around each call, and an
  agent's result carries `metrics.accumulated_usage`.

All three feed the same `RunMetrics`, and all three count tool calls through the shared tools.

## Takeaways

- Fix the definitions once, in shared code, then connect each framework to them.
- Count what you can outside the framework (tool calls); take only model calls from the framework.
- An eval is a fixed set of cases, scored automatically, re-run after every change.
- That completes the components. The capstones build ShipFast and the research agent in full.